In [2]:
# Fix SHRP_ID format mismatch
fwd['SHRP_ID'] = fwd['SHRP_ID'].astype(str).str.zfill(4)

# Keep only useful columns from EXP
exp_clean = exp[['SHRP_ID', 'STATE_CODE', 'CONSTRUCTION_NO', 
                 'PAVEMENT_FAMILY']].drop_duplicates()

# Drop useless FWD columns safely
columns_to_drop = ['PEAK_DEFL_8', 'PEAK_DEFL_9', 'NON_DECREASING_DEFL', 
                   'NON_DECREASING_DEFL_EXP', 'STATE_CODE_EXP', 
                   'LANE_NO_EXP', 'DROP_HEIGHT_EXP', 'DEFL_UNIT_ID']
fwd = fwd.drop(columns=columns_to_drop, errors='ignore')

# Merge
df2 = pd.merge(fwd, exp_clean, 
               on=['SHRP_ID', 'STATE_CODE', 'CONSTRUCTION_NO'], 
               how='inner')

# Drop any remaining nulls
df2 = df2.dropna()

print("Shape after merge and clean:", df2.shape)

Shape after merge and clean: (65534, 18)


In [3]:
# Encode categorical variables
le_pav = LabelEncoder()
df2['PAVEMENT_FAMILY_ENC'] = le_pav.fit_transform(df2['PAVEMENT_FAMILY'])

le_lane = LabelEncoder()
df2['LANE_NO_ENC'] = le_lane.fit_transform(df2['LANE_NO'])

# Define features for clustering
features2 = ['PEAK_DEFL_1', 'PEAK_DEFL_2', 'PEAK_DEFL_3', 
             'PEAK_DEFL_4', 'PEAK_DEFL_5', 'PEAK_DEFL_6', 
             'PEAK_DEFL_7', 'DROP_LOAD', 'DROP_HEIGHT',
             'PAVEMENT_FAMILY_ENC', 'LANE_NO_ENC']

X2 = df2[features2]

# Standardize features (Critical for K-Means)
scaler = StandardScaler()
X2_scaled = scaler.fit_transform(X2)

print("Features standardized and ready for clustering.")

Features standardized and ready for clustering.


In [4]:
# Train K-Means Clustering Model (Unsupervised)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df2['CLUSTER'] = kmeans.fit_predict(X2_scaled)

# Calculate average deflection per cluster to determine Good/Fair/Poor dynamically
df2['AVG_DEFL'] = df2[['PEAK_DEFL_1','PEAK_DEFL_2','PEAK_DEFL_3',
                       'PEAK_DEFL_4','PEAK_DEFL_5','PEAK_DEFL_6',
                       'PEAK_DEFL_7']].mean(axis=1)

# Sort clusters by average deflection
cluster_means = df2.groupby('CLUSTER')['AVG_DEFL'].mean().sort_values()

# Identify exact cluster indices
good_cluster_idx = cluster_means.index[0]
fair_cluster_idx = cluster_means.index[1]
poor_cluster_idx = cluster_means.index[2]

health_mapping = {
    good_cluster_idx: 'Good',
    fair_cluster_idx: 'Fair',
    poor_cluster_idx: 'Poor'
}

df2['HEALTH'] = df2['CLUSTER'].map(health_mapping)

# Calculate EXACT continuous structural health score using cluster distances
distances = kmeans.transform(X2_scaled)
dist_to_good = distances[:, good_cluster_idx]
dist_to_poor = distances[:, poor_cluster_idx]

# Formula: (Distance to Poor) / (Distance to Good + Distance to Poor) * 100
df2['FWD_SCORE'] = (dist_to_poor / (dist_to_good + dist_to_poor)) * 100

print("Cluster to Health Mapping:", health_mapping)
print("\\nDynamic Health Distribution:")
print(df2['HEALTH'].value_counts())
print("\\nSample Continuous FWD Scores:")
print(df2[['SHRP_ID', 'AVG_DEFL', 'HEALTH', 'FWD_SCORE']].head(10))

Cluster to Health Mapping: {np.int32(2): 'Good', np.int32(0): 'Fair', np.int32(1): 'Poor'}
\nDynamic Health Distribution:
HEALTH
Fair    25204
Good    21164
Poor    19166
Name: count, dtype: int64
\nSample Continuous FWD Scores:
  SHRP_ID    AVG_DEFL HEALTH  FWD_SCORE
0    0101  406.285714   Poor  23.659654
1    0101  201.571429   Fair  54.879028
2    0101  202.000000   Fair  54.891354
3    0101  202.714286   Fair  54.659099
4    0101  202.285714   Fair  54.819027
5    0101  286.142857   Poor  23.974308
6    0101  289.428571   Poor  23.462802
7    0101  289.857143   Poor  23.369921
8    0101  289.714286   Poor  23.436844
9    0101  382.857143   Poor  22.453255


In [5]:
# Calculate clustering evaluation metrics
sil_score = silhouette_score(X2_scaled, df2['CLUSTER'], sample_size=10000, random_state=42)
ch_score = calinski_harabasz_score(X2_scaled, df2['CLUSTER'])
db_score = davies_bouldin_score(X2_scaled, df2['CLUSTER'])

print("============================================")
print("     MODEL 2 CLUSTERING EVALUATION REPORT   ")
print("============================================")
print(f"Silhouette Score      : {sil_score:.4f}  | Range: -1 to 1 (closer to 1 is better)")
print(f"Calinski-Harabasz     : {ch_score:.2f} | Higher is better (measures density/separation)")
print(f"Davies-Bouldin Index  : {db_score:.4f}  | Closer to 0 is better (lower means distinct)")
print("============================================")

# Save models and preprocessors for the RHI script
joblib.dump(kmeans, '../models/fwd_kmeans_model.pkl')
joblib.dump(scaler, '../models/fwd_scaler.pkl')
joblib.dump(le_pav, '../models/fwd_le_pav.pkl')
joblib.dump(le_lane, '../models/fwd_le_lane.pkl')
joblib.dump(health_mapping, '../models/fwd_health_mapping.pkl')
print("\nUnsupervised model and preprocessors saved!")

     MODEL 2 CLUSTERING EVALUATION REPORT   
Silhouette Score      : 0.2864  | Range: -1 to 1 (closer to 1 is better)
Calinski-Harabasz     : 45953.49 | Higher is better (measures density/separation)
Davies-Bouldin Index  : 1.2538  | Closer to 0 is better (lower means distinct)

Unsupervised model and preprocessors saved!
